In [1]:
import sys
sys.path.append('/data/Aldhani/cv_fields/code/decode/')
sys.path.append('/data/Aldhani/cv_fields/code/')
sys.path.append('/data/Aldhani/cv_fields/code/smallholder-fields/code/')


In [2]:
import os
import cv2
import glob
import warnings
import numpy as np
import time
from osgeo import gdal
from osgeo import ogr, osr
from helpers_model import *

In [ ]:
# create instances from probability predictions

# define model and paths
model = 'fractal-resunet_finetune-v01'
prd_path = f'../preds/{model}'
ins_path = f'{prd_path}/instances/'
if not os.path.isfile(ins_path): os.makedirs(ins_path)

prd_fls = glob.glob(prd_path+'/*.tif')
print(len(prd_fls))

# extent and boundary threshold
t_ext = 0.4
t_bnd = 0.2

for f in prd_fls:
    out_file =  f'{ins_path}/{os.path.basename(f)[:-4]}_inst.tif'
    if not os.path.isfile(out_file):
        # predictions
        preds = gdal.Open(f).ReadAsArray()
        
        # instances
        ext = np.squeeze(preds[0])
        bnd = np.squeeze(preds[1])
        instances = InstSegm(ext/10000, bnd/10000, t_ext=t_ext, t_bound=t_bnd) 

        # map instances to better range for plotting
        unique_instances = sorted(np.unique(instances))
        if len(unique_instances)<=1: 
            instances_mapped = instances
            print('zero instances')
        if len(unique_instances)>1:
            min_instance = unique_instances[1]
            max_instance = unique_instances[-1]
            n_instances = len(unique_instances)
            instance_map = {x: i for i, x in enumerate(unique_instances)}
            def map_values(x):
                return instance_map[x]
            instances_mapped = np.array(list(map(map_values, instances.flatten()))).reshape( 
                instances.shape[0], instances.shape[1])
        
            # create memory copy of ds
            mem_ds = create_mem_ds(f, 1, dtype=gdal.GDT_UInt16)

            # write outputs to bands
            mem_ds.GetRasterBand(1).WriteArray(instances_mapped)
            
            # create physical copy of ds
            copy_mem_ds(out_file, mem_ds)

1


In [ ]:
# polygonize raster instances

# define model and paths
model = 'fractal-resunet_finetune-v01'
ins_path = f'../preds/{model}/instances'

# get list of instance rasters
prd_fls = glob.glob(ins_path+'/*.tif')
print(len(prd_fls))

# polygonize rasters to geopackages
for prd_fl in prd_fls:
  dst_layername = prd_fl[:-4]+'.gpkg'
  
  if not os.path.isfile(dst_layername):
    
    print(prd_fl)
    src_ds = gdal.Open(prd_fl)
    srcBand = src_ds.GetRasterBand(1)

    drv = ogr.GetDriverByName("GPKG")
    dst_ds = drv.CreateDataSource(dst_layername)
    dst_layer = dst_ds.CreateLayer(os.path.basename(dst_layername), srs = get_srs(src_ds))

    gdal.Polygonize(srcBand, srcBand, dst_layer, -1, [], callback=None)

    dst_layername = None
    src_ds = None
    srcBand = None
    drv = None
    dst_ds = None
    dst_layer = None

1
../preds/airbus_mozambique/instances/mozambique_tile_32636_00020_-0281_20230917_preds_inst.tif
